# 3 · Voz (Chatterbox Multilingual na GPU)

Gera uma narração em português a partir de um texto, com o [Chatterbox](https://github.com/resemble-ai/chatterbox) (licença MIT).
O texto é dividido em frases, cada uma gerada separadamente e unida com uma pausa curta (o modelo corta textos longos).
Grava `EditorVideos/HyperFrames/voz/<nome>.wav`.

Voz própria (opcional): ponha uma gravação limpa de 10 a 20 s em `HyperFrames/voz/referencia` e escolha o arquivo em VOZ_REF.
**Só a sua voz ou de quem autorizou.**

**Antes:** Ambiente de execução → Alterar tipo → **GPU T4**. Depois: Executar tudo. A instalação leva uns 5 min.

In [ ]:
#@title 1. Configuração
TEXTO = ""  #@param {type:"string"}
NOME = "narracao"  #@param {type:"string"}
VOZ_REF = ""  #@param {type:"string"}
#@markdown Vazio = voz padrão do modelo. Ou o nome de um arquivo em HyperFrames/voz/referencia.
EXPRESSAO = 0.5  #@param {type:"slider", min:0.25, max:1.5, step:0.05}
#@markdown 0,5 = natural. Mais alto = mais emoção e mais rápido.
RITMO = 0.5  #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown cfg_weight do modelo. Mais baixo = fala mais pausada.
PAUSA = 0.3  #@param {type:"number"}
SEMENTE = 7  #@param {type:"integer"}

# Pelo Colab CLI os valores chegam por variável de ambiente; o texto vem em base64 (acentos e quebras de linha intactos).
import os, base64
if os.environ.get('TEXTO_B64'): TEXTO = base64.b64decode(os.environ['TEXTO_B64']).decode('utf-8')
NOME = os.environ.get('NOME', NOME)
VOZ_REF = os.environ.get('VOZ_REF', VOZ_REF)
EXPRESSAO = float(os.environ.get('EXPRESSAO', EXPRESSAO))
RITMO = float(os.environ.get('RITMO', RITMO))

In [ ]:
#@title 2. Drive e status para o Painel
from google.colab import drive
import json, os, time, threading, subprocess, collections
from datetime import datetime

RAIZ = '/content/drive/MyDrive/EditorVideos'
STATUS_DIR = f'{RAIZ}/_colab/status'

# A pasta EditorVideos vem do PC (Drive para computador). Se a montagem do Colab estiver velha ela
# não aparece; remonta e, se ainda faltar, para aqui em vez de criar uma "EditorVideos (1)" vazia.
# Pelo Colab CLI (Painel/colab-cli) o Drive já vem montado por `colab drivemount`: drive.mount() aqui
# ficaria esperando um navegador que não existe.
if os.environ.get('EDITORVIDEOS_CLI') and os.path.isdir('/content/drive/MyDrive'):
    print('Drive montado pelo Colab CLI.')
else:
    drive.mount('/content/drive', force_remount=True)
    if not os.path.isdir(f'{RAIZ}/HyperFrames'):
        time.sleep(10)
        drive.flush_and_unmount()
        drive.mount('/content/drive', force_remount=True)
if not os.path.isdir(f'{RAIZ}/HyperFrames'):
    raise SystemExit('EditorVideos/HyperFrames não está visível neste Drive. Confira se o Colab está na mesma conta '
                     'do Drive do PC e se a sincronização terminou; depois Ambiente de execução → Desconectar e excluir.')

class Status:
    """Grava o andamento em _colab/status/<servico>.json. Batimento a cada 15 s enquanto roda."""
    def __init__(self, servico, notebook):
        os.makedirs(STATUS_DIR, exist_ok=True)
        self.arq = f'{STATUS_DIR}/{servico}.json'
        self.d = dict(servico=servico, notebook=notebook, estado='rodando', etapa='iniciando',
                      progresso=None, mensagem='', inicio=int(time.time() * 1000))
        self.log = collections.deque(maxlen=40)
        self._trava = threading.Lock()
        self._ultimo = 0
        self._vivo = True
        threading.Thread(target=self._batimento, daemon=True).start()
        self.gravar()

    def _gpu(self):
        try:
            out = subprocess.run(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total',
                                  '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=5).stdout
            nome, uso, usada, total = [x.strip() for x in out.strip().splitlines()[0].split(',')]
            return dict(nome=nome, uso=int(uso), memoria_usada_mb=int(usada), memoria_total_mb=int(total))
        except Exception:
            return None

    def _ram(self):
        try:
            m = {}
            for linha in open('/proc/meminfo'):
                k, v = linha.split(':')
                m[k] = int(v.split()[0])
            return dict(usada_mb=(m['MemTotal'] - m['MemAvailable']) // 1024, total_mb=m['MemTotal'] // 1024)
        except Exception:
            return None

    def gravar(self):
        with self._trava:
            self.d.update(log=list(self.log), gpu=self._gpu(), ram=self._ram(), atualizado=int(time.time() * 1000))
            tmp = self.arq + '.tmp'
            with open(tmp, 'w') as f:
                json.dump(self.d, f, ensure_ascii=False)
            os.replace(tmp, self.arq)
            self._ultimo = time.time()

    def _batimento(self):
        while self._vivo:
            time.sleep(15)
            if self._vivo:
                self.gravar()

    def __call__(self, etapa=None, progresso=None, mensagem=None, linha=None):
        mudou = etapa is not None and etapa != self.d['etapa']
        if etapa is not None: self.d['etapa'] = etapa
        if progresso is not None: self.d['progresso'] = round(progresso, 1)
        if mensagem is not None: self.d['mensagem'] = mensagem
        if linha:
            self.log.append(f'{datetime.now():%H:%M:%S} {linha}')
            print(linha)
        # Não martela o Drive: no máximo uma gravação a cada 3 s, salvo mudança de etapa.
        if mudou or linha or time.time() - self._ultimo > 3:
            self.gravar()

    def fim(self, estado='concluido', mensagem=''):
        self._vivo = False
        self.d.update(estado=estado, mensagem=mensagem, etapa='fim' if estado == 'concluido' else self.d['etapa'])
        if estado == 'concluido': self.d['progresso'] = 100
        self.gravar()

print('Drive montado. Status em', STATUS_DIR)

In [ ]:
#@title 3. Instalar o Chatterbox
st = Status('voz', '3_voz')
st('instalando', 2, 'chatterbox-tts 0.1.7', 'instalando o Chatterbox (uns 4 min)')
!pip -q install chatterbox-tts==0.1.7
# O chatterbox fixa outra versão do torch; o torchvision do Colab fica incompatível ("torchvision::nms does not
# exist") e quebra o transformers. A voz não usa torchvision: remover resolve.
!pip -q uninstall -y torchvision
st('instalado', 15, linha='Chatterbox instalado')

In [ ]:
#@title 4. Gerar a narração
import re, shutil, torch, torchaudio as ta

def frases(texto, limite=260):
    """Divide em frases e junta as curtas até ~limite caracteres; frase longa demais é quebrada nas vírgulas."""
    partes = [p.strip() for p in re.split(r'(?<=[.!?…])\s+|\n+', texto) if p.strip()]
    blocos, atual = [], ''
    for p in partes:
        pedacos = [p] if len(p) <= limite else [x.strip() for x in re.split(r'(?<=[,;:])\s+', p) if x.strip()]
        for x in pedacos:
            if atual and len(atual) + len(x) + 1 > limite:
                blocos.append(atual)
                atual = x
            else:
                atual = f'{atual} {x}'.strip()
    return blocos + ([atual] if atual else [])

VOZ_DIR = f'{RAIZ}/HyperFrames/voz'
# Pelo CLI o .wav fica na VM e o Painel baixa direto (sem esperar o Drive sincronizar); no navegador vai para o Drive.
DESTINO = '/content/voz' if os.environ.get('EDITORVIDEOS_CLI') else VOZ_DIR
try:
    NOME = re.sub(r'[^\w.-]+', '-', NOME.strip()) or 'narracao'
    blocos = frases(TEXTO)
    if not blocos:
        raise ValueError('TEXTO vazio: escreva o roteiro na configuração.')
    ref = f'{VOZ_DIR}/referencia/{VOZ_REF}' if VOZ_REF else None
    if ref and not os.path.exists(ref):
        raise FileNotFoundError(f'Voz de referência não encontrada: HyperFrames/voz/referencia/{VOZ_REF}')
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    gpu = torch.cuda.is_available()
    st('carregando modelo', 17, 'GPU' if gpu else 'CPU (muito lento!)', 'carregando o modelo')
    modelo = ChatterboxMultilingualTTS.from_pretrained(device='cuda' if gpu else 'cpu')
    torch.manual_seed(SEMENTE)
    pausa = torch.zeros(1, int(modelo.sr * PAUSA))
    partes = []
    for i, b in enumerate(blocos):
        st(f'gerando {i + 1}/{len(blocos)}', 20 + i / len(blocos) * 75, b[:80])
        extra = dict(audio_prompt_path=ref) if ref else {}
        wav = modelo.generate(b, language_id='pt', exaggeration=EXPRESSAO, cfg_weight=RITMO, **extra).cpu()
        partes += [wav, pausa]
    audio = torch.cat(partes[:-1], dim=-1)
    os.makedirs(DESTINO, exist_ok=True)
    ta.save(f'{DESTINO}/{NOME}.wav', audio, modelo.sr)
    seg = audio.shape[-1] / modelo.sr
    st(linha=f'ok: {NOME}.wav ({seg:.1f} s, {len(blocos)} trechos, voz {VOZ_REF or "padrão"})')
    st.fim('concluido', f'HyperFrames/voz/{NOME}.wav · {seg:.0f} s de fala')
except Exception as e:
    st(linha=f'ERRO: {e}')
    st.fim('erro', str(e))
    raise